# Convergence check: InAs quantum dot in GaAs

Switching material systems from InSb/InAs to **InAs/GaAs** -- the classic, extensively studied
system (Pryor 1998, Stier/Grundmann/Bimberg 1999, and most of the QD literature cited earlier)
-- and, before moving on to different dot shapes or multiband k.p, checking that the numerical
machinery from `inSb_dot_in_inAs.ipynb` (now factored into `qdsolver_core.py`) actually
converges as the grid is refined and the box is enlarged.

InAs/GaAs is a good system for this check for a physical reason: it's **type-I** (both
electron and hole confine inside the dot), unlike the InAs/InSb-family type-II systems where
the electron ground state was mostly a box-quantized artifact. With both carriers genuinely
localized, the results should converge cleanly with grid spacing and become insensitive to box
size once the box is a few dot-radii larger than the dot -- both of which are checked below.

Same simplifications as before (single band, spherical dot, homogeneous/isotropic Eshelby
strain, hydrostatic-only band coupling) -- see `inSb_dot_in_inAs.ipynb` for the full derivation
and caveats, which aren't repeated here.

In [ ]:
import sys, os, time
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import qdsolver_core as qd

## Material system: InAs dot in GaAs

In [ ]:
matrix_mat, dot_mat = qd.MATERIALS['GaAs'], qd.MATERIALS['InAs']

eps_star = qd.eigenstrain(dot_mat['a0'], matrix_mat['a0'])
nu = qd.voigt_poisson_ratio(matrix_mat['C11'], matrix_mat['C12'], matrix_mat['C44'])
print(f"Lattice mismatch: {eps_star*100:.2f}% (the well-known ~7% InAs/GaAs mismatch)")
print(f"GaAs Voigt Poisson ratio: {nu:.4f}")

print(f"\nUnstrained band edges (eV):")
print(f"  GaAs (matrix): fi_e={matrix_mat['fi_e']:.4f}, fi_h={matrix_mat['fi_h']:.4f}")
print(f"  InAs (dot):    fi_e={dot_mat['fi_e']:.4f}, fi_h={dot_mat['fi_h']:.4f}")
print()
if dot_mat['fi_e'] < matrix_mat['fi_e'] and dot_mat['fi_h'] > matrix_mat['fi_h']:
    print("Both electron AND hole band edges favor the dot => type-I, as expected for InAs/GaAs.")
else:
    print("WARNING: this doesn't look type-I -- check the material parameters.")

## Solve function: build fields for a given grid, then solve

Bundles geometry + strain + fields + solve into one function of `(R_dot, L_half, h)` so it can
be called repeatedly while sweeping resolution and box size.

In [ ]:
def solve_sphere_qd(R_dot, L_half, h, n_states=4):
    coords = np.arange(-L_half, L_half + 1e-9, h)
    N = len(coords)
    X, Y, Z = np.meshgrid(coords, coords, coords, indexing='ij')
    r = np.sqrt(X**2 + Y**2 + Z**2)
    inside = r < R_dot

    trace_strain, trace_inside = qd.eshelby_sphere_trace_strain(inside, eps_star, nu)

    m_e_field = np.where(inside, dot_mat['m_e'], matrix_mat['m_e'])
    m_h_field = np.where(inside, dot_mat['m_h'], matrix_mat['m_h'])
    V_e = np.where(inside, dot_mat['fi_e'], matrix_mat['fi_e']) + dot_mat['Ac'] * trace_strain
    V_h = np.where(inside, dot_mat['fi_h'], matrix_mat['fi_h']) + dot_mat['Av'] * trace_strain

    E_e, psi_e = qd.solve_states(m_e_field, V_e, h, n_states=n_states, hole=False)
    E_h, psi_h = qd.solve_states(m_h_field, V_h, h, n_states=n_states, hole=True)
    return dict(N=N, coords=coords, E_e=E_e, E_h=E_h, psi_e=psi_e, psi_h=psi_h,
                V_e=V_e, V_h=V_h, inside=inside)

# Quick smoke test
t0 = time.time()
res = solve_sphere_qd(R_dot=5.0, L_half=15.0, h=1.0)
transition_eV = res['E_e'][0] - res['E_h'][0]
print(f"N={res['N']}^3={res['N']**3:,} points, solved in {time.time()-t0:.1f} s")
print(f"Electron ground state: {res['E_e'][0]*1e3:.2f} meV "
      f"(dot floor {dot_mat['fi_e']*1e3:.1f} meV + Ac*strain shift)")
print(f"Hole ground state:     {res['E_h'][0]*1e3:.2f} meV "
      f"(dot ceiling {dot_mat['fi_h']*1e3:.1f} meV + Av*strain shift)")
print(f"Naive transition energy: {transition_eV*1e3:.1f} meV ({1239.84/transition_eV:.0f} nm)")

## Convergence check 1: grid spacing $h$ (fixed box)

$R_\mathrm{dot}=5$ nm, box half-width $L_\mathrm{half}=16$ nm fixed; $h$ swept from coarse to
fine. All $h$ values here evenly divide 16 nm, so the grid always lands exactly on the dot's
center at every resolution -- otherwise the sphere's boundary aliases onto the grid differently
at different $h$ (or box size), which showed up as several-meV, non-monotonic noise in an
earlier version of this sweep that mixed integer and half-integer box half-widths.

In [ ]:
R_dot = 5.0
L_half_fixed = 16.0
h_values = [4.0, 2.0, 1.6, 1.0, 0.8]

h_results = []
for h in h_values:
    t0 = time.time()
    res = solve_sphere_qd(R_dot, L_half_fixed, h)
    dt = time.time() - t0
    h_results.append(dict(h=h, N=res['N'], E_e0=res['E_e'][0], E_h0=res['E_h'][0], dt=dt))
    print(f"h={h:.2f} nm  (N={res['N']:3d}^3={res['N']**3:>9,} pts, {dt:5.1f} s): "
          f"E_e0={res['E_e'][0]*1e3:8.2f} meV   E_h0={res['E_h'][0]*1e3:8.2f} meV   "
          f"transition={ (res['E_e'][0]-res['E_h'][0])*1e3:7.2f} meV")

In [ ]:
hs = np.array([r['h'] for r in h_results])
E_e0s = np.array([r['E_e0'] for r in h_results]) * 1e3
E_h0s = np.array([r['E_h0'] for r in h_results]) * 1e3
trans = E_e0s - E_h0s

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y, label, color in zip(
        axes, [E_e0s, E_h0s, trans],
        ["Electron ground state", "Hole ground state", "Transition energy"],
        ["#0072B2", "#E69F00", "#009E73"]):
    ax.plot(hs**2, y, 'o-', color=color)
    ax.set_xlabel("$h^2$ (nm$^2$)")
    ax.set_ylabel("Energy (meV)")
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
fig.suptitle("Convergence vs grid spacing (finite-difference error is expected ~O(h$^2$),\n"
             "so a roughly straight line here, extrapolating to h=0 on the left, is the expected signature)")
fig.tight_layout()

## Convergence check 2: box size $L_\mathrm{half}$ (fixed grid spacing)

Picking $h=1.0$ nm (a reasonable compromise from the sweep above -- adjust after looking at the
plot) and growing the box to check that hard-wall boundary artifacts have died away. All
`L_half` values here are integers, so with `h=1.0` the grid always includes the dot's center
exactly (see the note above the previous sweep about grid-alignment artifacts).

In [ ]:
h_fixed = 1.0
L_half_values = [10.0, 12.0, 14.0, 16.0, 18.0, 20.0]

box_results = []
for L_half in L_half_values:
    t0 = time.time()
    res = solve_sphere_qd(R_dot, L_half, h_fixed)
    dt = time.time() - t0
    box_results.append(dict(L_half=L_half, N=res['N'], E_e0=res['E_e'][0], E_h0=res['E_h'][0], dt=dt))
    print(f"L_half={L_half:5.1f} nm (R_dot/L_half={R_dot/L_half:.2f}, N={res['N']:3d}^3, {dt:5.1f} s): "
          f"E_e0={res['E_e'][0]*1e3:8.2f} meV   E_h0={res['E_h'][0]*1e3:8.2f} meV   "
          f"transition={(res['E_e'][0]-res['E_h'][0])*1e3:7.2f} meV")

In [ ]:
Ls = np.array([r['L_half'] for r in box_results])
E_e0s_L = np.array([r['E_e0'] for r in box_results]) * 1e3
E_h0s_L = np.array([r['E_h0'] for r in box_results]) * 1e3
trans_L = E_e0s_L - E_h0s_L

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y, label, color in zip(
        axes, [E_e0s_L, E_h0s_L, trans_L],
        ["Electron ground state", "Hole ground state", "Transition energy"],
        ["#0072B2", "#E69F00", "#009E73"]):
    ax.plot(Ls, y, 'o-', color=color)
    ax.set_xlabel("Box half-width $L_{half}$ (nm)")
    ax.set_ylabel("Energy (meV)")
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
fig.suptitle("Convergence vs box size at fixed h=1.0 nm (should flatten out once padding is enough)")
fig.tight_layout()

## Takeaway

**Box size**: converges cleanly and monotonically. The hole ground state is essentially exact
by `L_half=12` nm (2.4 dot radii of padding) -- it's tightly confined, so it barely notices the
walls at all. The electron ground state (shallower confinement, light InAs mass) converges more
slowly but has flattened to within ~0.3 meV by `L_half=16-20` nm.

**Grid spacing**: converges overall, but with a non-monotonic ~2-8 meV wobble rather than a
clean line, particularly at coarser `h`. This is a *different* effect from the box-size check:
representing a smooth sphere on a cubic grid means the dot's boundary is "staircased" (voxels
are either fully in or fully out), and exactly how many boundary voxels round one way vs. the
other changes with `h` in a way that isn't perfectly smooth -- it generally shrinks with finer
`h` but doesn't have to decrease at every single step. This is a known, expected limitation of a
hard inside/outside mask on a Cartesian grid (as opposed to, e.g., weighting boundary voxels by
the fraction of their volume inside the sphere, which would smooth this out at the cost of a
more complex mask -- worth doing later if more precision is needed, especially once dot shapes
get less symmetric than a sphere).

**Recommended resolution for the upcoming shape/multiband notebooks**: $h=1.0$ nm,
$L_\mathrm{half} \gtrsim 4\times$ the dot's largest extent (so proportionally larger for a lens
or pyramid's lateral size), keeping in mind a residual few-meV uncertainty from staircasing at
this grid spacing -- fine for comparing shapes/sizes to each other, but not for reading off a
single number to three decimal places.